In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=""  # put your real key here
)

class JokeState(TypedDict):
    topic: str
    joke: str
    explain: str

def joke(state: JokeState):
    prompt = f"Generate me a joke on the topic {state['topic']}"
    response = llm.invoke(prompt)
    return {"joke": response.content}

def explain(state: JokeState):
    prompt = f"Explain me the joke {state['joke']}"
    response = llm.invoke(prompt)
    return {"explain": response.content}

graph = StateGraph(JokeState)
graph.add_node("joke", joke)
graph.add_node("explain", explain)
graph.add_edge(START, "joke")
graph.add_edge("joke", "explain")
graph.add_edge("explain", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

# checkpointer is an object of InMemorySaver()
# when we compile the graph, we pass the checkpointer
# this tells the graph to save state even after execution ends,
# including intermediate values — basically the full conversation/history

config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "chesses"}, config=config1)
workflow.get_state(config1)
list(workflow.get_state(config1))

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({"topic": "education_system"}, config=config2)

{'topic': 'education_system',
 'joke': 'Why did the student bring a ladder to school?\n\nBecause they wanted to reach their full potential in the education system, but it turns out they just needed to take a step in the right direction.',
 'explain': 'A clever play on words. The joke is a pun, using the multiple meanings of "reach" and "step" to create a humorous effect.\n\nThe setup "Why did the student bring a ladder to school?" primes the listener to expect a literal reason for bringing a ladder, such as to reach a high shelf. The phrase "reach their full potential" is a common idiomatic expression meaning to achieve one\'s maximum capabilities or goals.\n\nThe punchline subverts this expectation by taking the phrase "reach their full potential" literally, implying that the student needs a ladder to physically reach something. However, the second part of the punchline "but it turns out they just needed to take a step in the right direction" is a wordplay on the phrase "take a step i